In [1]:
# ─────────────────────────── standard libs ────────────────────────────────
import os

# FIX 1: Set expandable segments BEFORE any CUDA allocations
# This reduces memory fragmentation and can recover hundreds of MB
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

from pathlib import Path
from collections import Counter
import logging
import warnings
import math

# ────────────────────────────── data stack ────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt

# ────────────────────────────── sklearn ───────────────────────────────────
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix,
)
from sklearn.model_selection import train_test_split

# ─────────────────────────────── torch ────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, TensorDataset, DataLoader
from torch.utils.checkpoint import checkpoint   # FIX 2: gradient checkpointing

# ─────────────────────────────── other ────────────────────────────────────
import mne
from tqdm import tqdm

mne.set_log_level("ERROR")
logging.getLogger("mne").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=RuntimeWarning)

print("✅  Imports done. PYTORCH_ALLOC_CONF =", os.environ.get("PYTORCH_ALLOC_CONF"))

✅  Imports done. PYTORCH_ALLOC_CONF = expandable_segments:True


In [2]:
edf_path = '/kaggle/input/datasets/iamlokigodofmischief/tuh-eeg/tuh_eeg_epilepsy/tuh_eeg_epilepsy/00_epilepsy/aaaaaanr/s001_2003/02_tcp_le/aaaaaanr_s001_t001.edf'
raw = mne.io.read_raw_edf(edf_path, preload=False, verbose=False)
print(raw.info['sfreq'], 'Hz,', len(raw.ch_names), 'channels,',
      raw.n_times / raw.info['sfreq'] / 60, 'minutes long')

250.0 Hz, 33 channels, 20.233333333333334 minutes long


In [3]:
"""
One-pass census for the TUH EEG Epilepsy subset
"""

ROOT = Path("/kaggle/input/datasets/iamlokigodofmischief/tuh-eeg/tuh_eeg_epilepsy/tuh_eeg_epilepsy")

def clean_label(ch):
    if ch.startswith("EEG "):
        ch = ch.split()[1]
    return ch.split("-")[0]

count_hist = Counter()
rate_hist  = Counter()
label_hist = Counter()
tot_files  = 0

for edf in ROOT.rglob("*.edf"):
    raw = mne.io.read_raw_edf(edf, preload=False, verbose=False)
    tot_files += 1
    count_hist[len(raw.ch_names)] += 1
    rate_hist[raw.info["sfreq"]] += 1
    for ch in raw.ch_names:
        label_hist[clean_label(ch)] += 1

print(f"\nScanned {tot_files:,} EDF files\n")

def pretty(counter, idx_name, col_name, numeric=False):
    df = (pd.Series(counter)
            .rename_axis(idx_name)
            .reset_index(name=col_name))
    if numeric:
        df[idx_name] = pd.to_numeric(df[idx_name])
        df = df.sort_values(idx_name)
    else:
        df = df.sort_values(col_name, ascending=False)
    return df.reset_index(drop=True)

df_counts = pretty(count_hist, "n_channels", "n_files", numeric=True)
print("Files per channel-count\n" + df_counts.to_string(index=False))

df_rates  = pretty(rate_hist , "Hz", "n_files", numeric=True)
print("\nSampling-rate distribution\n" + df_rates.to_string(index=False))

df_labels = (pretty(label_hist, "channel", "n_files")
               .assign(percent=lambda d: 100*d.n_files/tot_files))
print("\nChannel presence (top 30)\n" +
      df_labels.head(30).to_string(index=False,
                                   formatters={"percent": "{:.2f}%".format}))


Scanned 2,298 EDF files

Files per channel-count
 n_channels  n_files
         17        2
         18        6
         27      158
         28      106
         29      101
         30      229
         31      179
         32      203
         33      260
         34      595
         35       19
         36      314
         41      126

Sampling-rate distribution
    Hz  n_files
 250.0      681
 256.0     1354
 400.0       75
 512.0      168
1000.0       20

Channel presence (top 30)
channel  n_files percent
    FP1     2298 100.00%
    FP2     2298 100.00%
     F3     2298 100.00%
     F4     2298 100.00%
     C3     2298 100.00%
     C4     2298 100.00%
     P3     2298 100.00%
     P4     2298 100.00%
     O1     2298 100.00%
     O2     2298 100.00%
     F7     2298 100.00%
     F8     2298 100.00%
     T3     2298 100.00%
     T4     2298 100.00%
     T5     2298 100.00%
     T6     2298 100.00%
     CZ     2298 100.00%
     PZ     2290  99.65%
     FZ     2290  99.65%
     

In [4]:
ROOT = Path("/kaggle/input/datasets/iamlokigodofmischief/tuh-eeg/tuh_eeg_epilepsy/tuh_eeg_epilepsy")

def build_index(root: Path) -> pd.DataFrame:
    rows = []
    for class_dir in ('00_epilepsy', '01_no_epilepsy'):
        label = 'epilepsy' if class_dir.startswith('00') else 'control'
        group_path = root / class_dir
        for patient_dir in group_path.iterdir():
            pid = patient_dir.name
            for session_dir in patient_dir.iterdir():
                sid = session_dir.name
                for montage_dir in session_dir.iterdir():
                    mid = montage_dir.name
                    for edf in montage_dir.glob('*.edf'):
                        rows.append({
                            'label'   : label,
                            'patient' : pid,
                            'session' : sid,
                            'montage' : mid,
                            'edf_path': str(edf)
                        })
    return pd.DataFrame(rows)

df = build_index(ROOT)
print(f"Indexed {len(df):,} EDF files")
print(df.head())

Indexed 2,298 EDF files
      label   patient    session    montage  \
0  epilepsy  aaaaakvr  s001_2010  01_tcp_ar   
1  epilepsy  aaaaaklv  s003_2011  01_tcp_ar   
2  epilepsy  aaaaaklv  s002_2011  01_tcp_ar   
3  epilepsy  aaaaaklv  s001_2010  02_tcp_le   
4  epilepsy  aaaaaawu  s004_2011  01_tcp_ar   

                                            edf_path  
0  /kaggle/input/datasets/iamlokigodofmischief/tu...  
1  /kaggle/input/datasets/iamlokigodofmischief/tu...  
2  /kaggle/input/datasets/iamlokigodofmischief/tu...  
3  /kaggle/input/datasets/iamlokigodofmischief/tu...  
4  /kaggle/input/datasets/iamlokigodofmischief/tu...  


In [5]:
df['label'].value_counts()
df.groupby('patient').size()
df[df['label']=='epilepsy'].head()

,label,patient,session,montage,edf_path
0,epilepsy,aaaaakvr,s001_2010,01_tcp_ar,/kaggle/input/datasets/iamlokigodofmischief/tu...
1,epilepsy,aaaaaklv,s003_2011,01_tcp_ar,/kaggle/input/datasets/iamlokigodofmischief/tu...
2,epilepsy,aaaaaklv,s002_2011,01_tcp_ar,/kaggle/input/datasets/iamlokigodofmischief/tu...
3,epilepsy,aaaaaklv,s001_2010,02_tcp_le,/kaggle/input/datasets/iamlokigodofmischief/tu...
4,epilepsy,aaaaaawu,s004_2011,01_tcp_ar,/kaggle/input/datasets/iamlokigodofmischief/tu...


In [6]:
# ─── config ──────────────────────────────────────────────────────────────────
NUM_PATIENTS = 60
ROOT         = Path("/kaggle/input/datasets/iamlokigodofmischief/tuh-eeg/"
                    "tuh_eeg_epilepsy/tuh_eeg_epilepsy")

TARGET_CHANS = ['FP1','FP2','F3','F4','C3','C4','P3','P4','O1','O2',
                'F7','F8','T3','T4','T5','T6','FZ','CZ','PZ','A1','A2']
MIN_CHANS    = 16

FS_TARGET    = 128
WIN_SEC      = 4
WIN_SAMPLES  = FS_TARGET * WIN_SEC          # 512
AMP_THRESH   = 150e-6
STEP_SEC     = 2
STEP_SAMPLES = FS_TARGET * STEP_SEC         # 256


def patient_total_bytes(patient_dir: Path) -> int:
    return sum(edf.stat().st_size for edf in patient_dir.rglob("*.edf"))

def clean_name(ch: str) -> str:
    if ch.startswith("EEG "):
        ch = ch.split()[1]
    return ch.split("-")[0]


def tidy_raw(raw: mne.io.BaseRaw) -> mne.io.BaseRaw | None:
    raw.rename_channels({c: clean_name(c) for c in raw.ch_names}, verbose=False)
    available = [c for c in TARGET_CHANS if c in raw.ch_names]
    if len(available) < MIN_CHANS:
        return None
    raw.pick(available)
    raw.reorder_channels(sorted(available, key=TARGET_CHANS.index))
    raw.set_eeg_reference("average", verbose=False)
    raw.filter(1., 40., fir_design="firwin", verbose=False)
    raw.notch_filter(60., fir_design="firwin", verbose=False)
    if raw.info["sfreq"] != FS_TARGET:
        raw.resample(FS_TARGET, npad="auto", verbose=False)
    return raw


def load_windows(edf_path: Path):
    try:
        raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
    except Exception:
        return [], []

    raw = tidy_raw(raw)
    if raw is None:
        return [], []

    data = raw.get_data().astype(np.float32)
    n_ch, n_times = data.shape
    stem = edf_path.stem

    windows, names = [], []
    for start in range(0, n_times - WIN_SAMPLES + 1, STEP_SAMPLES):
        win = data[:, start: start + WIN_SAMPLES]
        if np.any(np.abs(win) > AMP_THRESH):
            continue
        mu  = win.mean(axis=1, keepdims=True)
        std = win.std(axis=1, keepdims=True) + 1e-8
        win = (win - mu) / std
        windows.append(win)
        names.append(f"{stem}_{start//FS_TARGET:07d}s")

    return windows, names


ep_patients = sorted((ROOT / "00_epilepsy").iterdir(),    key=patient_total_bytes)
ne_patients = sorted((ROOT / "01_no_epilepsy").iterdir(), key=patient_total_bytes)

ep_sizes = [(p, patient_total_bytes(p)) for p in ep_patients]
ne_sizes = [(p, patient_total_bytes(p)) for p in ne_patients]

pairs, used_ne = [], set()
for ep_path, ep_sz in ep_sizes:
    best = None
    for ne_path, ne_sz in ne_sizes:
        if ne_path in used_ne:
            continue
        diff = abs(ep_sz - ne_sz)
        if best is None or diff < best[1]:
            best = (ne_path, diff)
    if best is None:
        break
    used_ne.add(best[0])
    pairs.append((ep_path, best[0]))
    if len(pairs) == NUM_PATIENTS:
        break

ep_sel = [p for p, _ in pairs]
ne_sel = [p for _, p in pairs]

all_windows, all_names, all_labels = [], [], []

def process_group(group, lab):
    pbar = tqdm(group, desc=f"{lab}", unit="patient")
    for patient_dir in pbar:
        for edf in patient_dir.rglob("*.edf"):
            wins, names = load_windows(edf)
            all_windows.extend(wins)
            all_names.extend(names)
            all_labels.extend([lab] * len(wins))
        pbar.set_postfix(windows=len(all_windows))

process_group(ep_sel, "epilepsy")
process_group(ne_sel, "non-epilepsy")

N_CH = len(TARGET_CHANS)   # 21

def pad_to_target(win):
    n = win.shape[0]
    if n == N_CH:
        return win
    out = np.zeros((N_CH, WIN_SAMPLES), dtype=np.float32)
    out[:n] = win
    return out

X_raw  = np.stack([pad_to_target(w) for w in all_windows]).astype(np.float32)
y_raw  = np.array([1 if l == "epilepsy" else 0 for l in all_labels], dtype=np.int64)
names  = np.array(all_names)

print(f"\n✅  Total windows : {len(X_raw):,}")
print(f"   epilepsy      : {(y_raw==1).sum():,}")
print(f"   non-epilepsy  : {(y_raw==0).sum():,}")
print(f"   X shape       : {X_raw.shape}   dtype: {X_raw.dtype}")

non-epilepsy: 100%|██████████| 60/60 [03:56<00:00,  3.94s/patient, windows=183795]



✅  Total windows : 183,795
   epilepsy      : 84,331
   non-epilepsy  : 99,464
   X shape       : (183795, 21, 512)   dtype: float32


In [7]:
# ─── patient-level split (70 / 15 / 15) ──────────────────────────────────────
patient_ids = np.array([n.split("_")[0] for n in names])

patients_ep = sorted(set(patient_ids[y_raw == 1]))
patients_ne = sorted(set(patient_ids[y_raw == 0]))

rng = np.random.default_rng(seed=42)
rng.shuffle(patients_ep);  rng.shuffle(patients_ne)

def split_ids(ids):
    n = len(ids)
    n_tr = int(round(0.70 * n));  n_va = int(round(0.15 * n))
    return list(ids[:n_tr]), list(ids[n_tr:n_tr+n_va]), list(ids[n_tr+n_va:])

ep_train, ep_val, ep_test = split_ids(patients_ep)
ne_train, ne_val, ne_test = split_ids(patients_ne)

train_pids = set(ep_train + ne_train)
val_pids   = set(ep_val   + ne_val)
test_pids  = set(ep_test  + ne_test)

tr_mask   = np.array([p in train_pids for p in patient_ids])
val_mask  = np.array([p in val_pids   for p in patient_ids])
test_mask = np.array([p in test_pids  for p in patient_ids])

X_train, y_train = X_raw[tr_mask],   y_raw[tr_mask]
X_val,   y_val   = X_raw[val_mask],  y_raw[val_mask]
X_test,  y_test  = X_raw[test_mask], y_raw[test_mask]

def summary(X, y, name):
    print(f"  {name:>6}: {len(X):>6,} windows  "
          f"| epi={y.sum():,}  non={(y==0).sum():,}")

print("📊  Split summary")
summary(X_train, y_train, "train")
summary(X_val,   y_val,   "val")
summary(X_test,  y_test,  "test")

from torch.utils.data import WeightedRandomSampler

# FIX 3: Reduce batch size to 16 (was 32).
# The per-sample attention in STM/TTM is O(S^2 * C^2) — 32 samples
# simultaneously fills ~77 GB. 16 gives comfortable headroom.
# Effective batch size is preserved by gradient accumulation in training.
BATCH_SIZE = 16

def make_loader(X, y, weighted=False):
    ds = TensorDataset(
        torch.from_numpy(X),
        torch.from_numpy(y),
    )
    if weighted:
        counts  = np.bincount(y)
        weights = 1.0 / counts[y]
        sampler = WeightedRandomSampler(
            torch.from_numpy(weights).float(),
            num_samples=len(weights), replacement=True
        )
        return DataLoader(ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=2, pin_memory=True)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=2, pin_memory=True)

train_loader = make_loader(X_train, y_train, weighted=True)
val_loader   = make_loader(X_val,   y_val)
test_loader  = make_loader(X_test,  y_test)

xb, yb = next(iter(train_loader))
print(f"\n✅  Batch — X: {xb.shape}  y: {yb.shape}  dtype: {xb.dtype}")
print(f"   Class balance in batch: epi={yb.sum().item()}  non={BATCH_SIZE - yb.sum().item()}")

📊  Split summary
   train: 124,992 windows  | epi=55,985  non=69,007
     val: 32,944 windows  | epi=14,438  non=18,506
    test: 25,859 windows  | epi=13,908  non=11,951

✅  Batch — X: torch.Size([16, 21, 512])  y: torch.Size([16])  dtype: torch.float32
   Class balance in batch: epi=8  non=8


In [8]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

# ─────────────────────────────────────────────────────────────────────────────
# EEGformer components
# ─────────────────────────────────────────────────────────────────────────────

def trunc_normal(tensor, mean=0., std=1., a=-2., b=2.):
    def norm_cdf(x):
        return (1. + math.erf(x / math.sqrt(2.))) / 2.
    with torch.no_grad():
        l = norm_cdf((a - mean) / std)
        u = norm_cdf((b - mean) / std)
        tensor.uniform_(2 * l - 1, 2 * u - 1)
        tensor.erfinv_()
        tensor.mul_(std * math.sqrt(2.))
        tensor.add_(mean)
        tensor.clamp_(min=a, max=b)
        return tensor


class Mlp(nn.Module):
    def __init__(self, in_features, hidden_features=None, out_features=None,
                 act_layer=nn.GELU, drop=0.):
        super().__init__()
        out_features     = out_features or in_features
        hidden_features  = hidden_features or in_features
        self.fc1  = nn.Linear(in_features, hidden_features)
        self.act  = act_layer()
        self.fc2  = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        return self.drop(self.fc2(self.drop(self.act(self.fc1(x)))))


class GenericTFB(nn.Module):
    """Transformer block used in RTM and STM."""
    def __init__(self, emb_size, num_heads):
        super().__init__()
        self.M  = emb_size
        self.hA = num_heads
        self.Dh = emb_size // num_heads
        self.Wqkv = nn.Parameter(torch.randn(3, num_heads, self.Dh, emb_size))
        self.Wo   = nn.Parameter(torch.randn(emb_size, emb_size))
        self.ln1  = nn.LayerNorm(emb_size)
        self.ln2  = nn.LayerNorm(emb_size)
        self.mlp  = Mlp(emb_size, emb_size * 4)

    def forward(self, x, z):
        qkv = torch.einsum('xhdm,ijm->xijhd', self.Wqkv, self.ln1(z))
        q, k, v = qkv[0], qkv[1], qkv[2]
        att = (q.transpose(1, 2) / math.sqrt(self.Dh)) @ k.transpose(1, 2).transpose(-2, -1)
        att = torch.softmax(att, dim=-1)
        imv = (att @ v.transpose(1, 2)).transpose(1, 2)
        z = torch.einsum('nm,ijm->ijn', self.Wo,
                         imv.reshape(z.shape[0], z.shape[1], self.M)) + z
        z = self.mlp(self.ln2(z)) + z
        return z


class TemporalTFB(nn.Module):
    def __init__(self, emb_size, num_heads, avgf):
        super().__init__()
        self.M    = emb_size
        self.hA   = num_heads
        self.Dh   = emb_size // num_heads
        self.avgf = avgf
        self.Wqkv = nn.Parameter(torch.randn(3, num_heads, self.Dh, emb_size))
        self.Wo   = nn.Parameter(torch.randn(emb_size, emb_size))
        self.ln1  = nn.LayerNorm(emb_size)
        self.ln2  = nn.LayerNorm(emb_size)
        self.mlp  = Mlp(emb_size, emb_size * 4)

    def forward(self, x, z):
        qkv = torch.einsum('xhdm,im->xihd', self.Wqkv, self.ln1(z))
        q, k, v = qkv[0], qkv[1], qkv[2]
        att = (q.transpose(0, 1) / math.sqrt(self.Dh)) @ k.transpose(0, 1).transpose(-2, -1)
        att = torch.softmax(att, dim=-1)
        imv = (att @ v.transpose(0, 1)).transpose(0, 1)
        z = torch.einsum('nm,im->in', self.Wo,
                         imv.reshape(self.avgf + 1, self.M)) + z
        z = self.mlp(self.ln2(z)) + z
        return z


class ODCM(nn.Module):
    def __init__(self, input_channels, kernel_size=10):
        super().__init__()
        self.ncf = 120
        C = input_channels
        self.cv1  = nn.Conv1d(C, C,            kernel_size, groups=C, padding='valid')
        self.cv2  = nn.Conv1d(C, C,            kernel_size, groups=C, padding='valid')
        self.cv3  = nn.Conv1d(C, self.ncf * C, kernel_size, groups=C, padding='valid')
        self.relu = nn.ReLU()

    def forward(self, x):           # (C, T)
        x = self.relu(self.cv1(x))
        x = self.relu(self.cv2(x))
        x = self.relu(self.cv3(x))
        S = x.shape[1]
        C = x.shape[0] // self.ncf
        return x.reshape(C, self.ncf, S)   # (C, D, S)


class RTM(nn.Module):
    def __init__(self, odcm_out_shape, num_blocks, num_heads):
        super().__init__()
        C, D, S = odcm_out_shape
        self.M  = D
        self.tK = num_blocks
        self.weight = nn.Parameter(torch.randn(D, D))
        self.bias   = nn.Parameter(torch.zeros(S, C + 1, D))
        self.cls    = nn.Parameter(torch.zeros(S, 1, D))
        trunc_normal(self.bias, std=.02)
        trunc_normal(self.cls,  std=.02)
        self.tfbs = nn.ModuleList([GenericTFB(D, num_heads) for _ in range(num_blocks)])

    def forward(self, x):           # x : (C, D, S)
        x = x.permute(2, 0, 1)                              # (S, C, D)
        z = torch.einsum('lm,ijm->ijl', self.weight, x)     # (S, C, D)
        z = torch.cat([self.cls, z], dim=1)                  # (S, C+1, D)
        z = z + self.bias
        for tfb in self.tfbs:
            z = tfb(x, z)
        return z                                             # (S, C+1, D)


class STM(nn.Module):
    def __init__(self, rtm_out_shape, num_blocks, num_heads):
        super().__init__()
        S, Cp1, D = rtm_out_shape
        self.M  = D
        self.tK = num_blocks
        self.weight = nn.Parameter(torch.randn(D, D))
        self.bias   = nn.Parameter(torch.zeros(Cp1, S + 1, D))
        self.cls    = nn.Parameter(torch.zeros(Cp1, 1, D))
        trunc_normal(self.bias, std=.02)
        trunc_normal(self.cls,  std=.02)
        self.tfbs = nn.ModuleList([GenericTFB(D, num_heads) for _ in range(num_blocks)])

    def forward(self, x):           # x : (S, C+1, D)
        x = x.transpose(0, 1)                               # (C+1, S, D)
        z = torch.einsum('lm,ijm->ijl', self.weight, x)     # (C+1, S, D)
        z = torch.cat([self.cls, z], dim=1)                  # (C+1, S+1, D)
        z = z + self.bias
        for tfb in self.tfbs:
            z = tfb(x, z)
        return z                                             # (C+1, S+1, D)


class TTM(nn.Module):
    def __init__(self, stm_out_shape, num_submatrices, num_blocks, num_heads):
        super().__init__()
        Cp1, Sp1, D = stm_out_shape
        self.avgf = num_submatrices
        self.M    = num_submatrices
        self.tK   = num_blocks
        assert D % num_submatrices == 0, f"D={D} must be divisible by num_submatrices={num_submatrices}"
        assert self.M % num_heads == 0,  f"M={self.M} must be divisible by num_heads={num_heads}"

        flat = Cp1 * Sp1
        self.weight = nn.Parameter(torch.randn(self.M, flat))
        self.bias   = nn.Parameter(torch.zeros(num_submatrices + 1, self.M))
        self.cls    = nn.Parameter(torch.zeros(1, self.M))
        trunc_normal(self.bias, std=.02)
        trunc_normal(self.cls,  std=.02)
        self.tfbs = nn.ModuleList([
            TemporalTFB(self.M, num_heads, num_submatrices)
            for _ in range(num_blocks)
        ])
        self.ln = nn.LayerNorm(self.M)
        self._Cp1, self._Sp1, self._D = Cp1, Sp1, D

    def forward(self, x):           # x : (C+1, S+1, D)
        Cp1, Sp1, D = x.shape
        seg  = D // self.avgf
        flat = Cp1 * Sp1

        xc  = x.permute(2, 0, 1)                                # (D, C+1, S+1)
        xc  = xc.reshape(self.avgf, seg, Cp1, Sp1).mean(dim=1)  # (M, C+1, S+1)
        alt = xc.reshape(self.avgf, flat)                        # (M, flat)

        z = torch.einsum('lm,im->il', self.weight, alt)          # (M, M)
        z = torch.cat([self.cls, z], dim=0)                      # (M+1, M)
        z = z + self.bias
        for tfb in self.tfbs:
            z = tfb(x, z)
        z = self.ln(z)                                           # (M+1, M)

        out = torch.einsum('tm,mf->tf', z, self.weight)          # (M+1, flat)
        return out.reshape(self.avgf + 1, Cp1, Sp1)              # (M+1, C+1, S+1)


class CNNDecoder(nn.Module):
    def __init__(self, ttm_out_shape, num_classes, CF_second=2):
        super().__init__()
        Mp1, Cp1, Sp1 = ttm_out_shape
        self.cvd1 = nn.Conv1d(Cp1,  1,         kernel_size=1)
        self.cvd2 = nn.Conv1d(Sp1,  CF_second, kernel_size=1)
        self.cvd3 = nn.Conv1d(Mp1,  Mp1 // 2,  kernel_size=1)
        self.fc   = nn.Linear((Mp1 // 2) * CF_second, num_classes)
        self.relu = nn.ReLU()

    def forward(self, x):           # x : (M+1, C+1, S+1)
        x = x.permute(2, 1, 0)                                                      # (S+1, C+1, M+1)
        x = self.relu(self.cvd1(x))                                                  # (S+1, 1,   M+1)
        x = x.squeeze(1)                                                             # (S+1, M+1)
        x = self.relu(self.cvd2(x.unsqueeze(0))).squeeze(0).transpose(0, 1)         # (M+1, CF_second)
        x = self.relu(self.cvd3(x.unsqueeze(0))).squeeze(0)                         # (M//2+1, CF_second)
        x = self.fc(x.reshape(1, -1))                                                # (1, num_classes)
        return x


# ─── L_SCLNet with EEGformer backbone ────────────────────────────────────────
class L_SCLNet_EEGformer(nn.Module):
    """
    L_SCLNet redesigned with an EEGformer backbone.
    Pipeline: ODCM → RTM → STM → TTM → CNNDecoder

    FIX 4: forward() now uses gradient checkpointing on the two most
    memory-intensive stages (STM and TTM). This recomputes activations
    during backward instead of storing them, cutting peak VRAM by ~35%.
    """
    def __init__(
        self,
        input_channels  = 21,
        time_steps      = 512,
        num_classes     = 2,
        kernel_size     = 10,
        num_blocks      = 3,
        num_heads_rtm   = 6,
        num_heads_stm   = 6,
        num_heads_ttm   = 5,
        num_submatrices = 10,
        CF_second       = 2,
        use_checkpoint  = True,   # ← new flag; set False to disable
    ):
        super().__init__()
        ncf = 120
        C   = input_channels
        D   = ncf
        S   = time_steps - 3 * (kernel_size - 1)

        assert D % num_heads_rtm == 0
        assert D % num_heads_stm == 0
        assert num_submatrices % num_heads_ttm == 0
        assert D % num_submatrices == 0

        self.use_checkpoint = use_checkpoint
        self.odcm    = ODCM(C, kernel_size)
        self.rtm     = RTM((C, D, S),       num_blocks, num_heads_rtm)
        self.stm     = STM((S, C+1, D),     num_blocks, num_heads_stm)
        self.ttm     = TTM((C+1, S+1, D),   num_submatrices, num_blocks, num_heads_ttm)
        self.decoder = CNNDecoder((num_submatrices+1, C+1, S+1), num_classes, CF_second)

    def _forward_single(self, xi):
        """Process one sample (C, T) → (1, num_classes)."""
        xi = self.odcm(xi)      # (C, D, S)
        xi = self.rtm(xi)       # (S, C+1, D)
        # FIX 4a: checkpoint STM — most memory-hungry stage
        if self.use_checkpoint and self.training:
            xi = checkpoint(self.stm, xi, use_reentrant=False)
        else:
            xi = self.stm(xi)
        # FIX 4b: checkpoint TTM — second most memory-hungry stage
        if self.use_checkpoint and self.training:
            xi = checkpoint(self.ttm, xi, use_reentrant=False)
        else:
            xi = self.ttm(xi)
        xi = self.decoder(xi)   # (1, num_classes)
        return xi

    def forward(self, x):       # x : (B, C, T)
        outs = []
        for i in range(x.shape[0]):
            outs.append(self._forward_single(x[i]))
        return torch.cat(outs, dim=0)   # (B, num_classes)


print("✅  Model classes defined.")

✅  Model classes defined.


In [9]:
# ═══════════════════════════════════════════════════════════════════════════
# SELF-SUPERVISED CONTRASTIVE PRE-TRAINING  (Paper §2.3 – §2.4)
# ═══════════════════════════════════════════════════════════════════════════

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
from tqdm import tqdm


# ─── 1.  Data augmentations ──────────────────────────────────────────────────

def augment_gaussian(x: torch.Tensor,
                     alpha: float = 0.0,
                     beta:  float = 0.1) -> torch.Tensor:
    return x + torch.randn_like(x) * beta + alpha


def augment_flip(x: torch.Tensor) -> torch.Tensor:
    return torch.flip(x, dims=[-1])


# ─── 2.  Contrastive Dataset ─────────────────────────────────────────────────

class ContrastiveEEGDataset(Dataset):
    def __init__(self, X: np.ndarray,
                 noise_alpha: float = 0.0,
                 noise_beta:  float = 0.1):
        self.X = torch.from_numpy(X)
        self.alpha = noise_alpha
        self.beta  = noise_beta

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x  = self.X[idx]
        v1 = augment_gaussian(x, self.alpha, self.beta)
        v2 = augment_flip(x)
        return v1, v2


# ─── 3.  Backbone feature extractor ──────────────────────────────────────────

class EEGformerBackbone(nn.Module):
    """
    Wraps the four transformer stages of L_SCLNet_EEGformer.
    Returns a flat feature vector per sample.

    FIX 5: forward_single delegates to the full model's _forward_single
    (which already uses gradient checkpointing on STM and TTM).
    The sample-wise Python loop is kept because the stages are inherently
    sample-wise; what matters is that each sample's graph is freed before
    the next is computed — checkpointing handles that.
    """
    def __init__(self, full_model):
        super().__init__()
        self.odcm = full_model.odcm
        self.rtm  = full_model.rtm
        self.stm  = full_model.stm
        self.ttm  = full_model.ttm
        self.use_checkpoint = full_model.use_checkpoint

    def forward_single(self, xi):
        """Process one sample (C, T) → flat feature vector."""
        xi = self.odcm(xi)
        xi = self.rtm(xi)
        if self.use_checkpoint and self.training:
            xi = checkpoint(self.stm, xi, use_reentrant=False)
        else:
            xi = self.stm(xi)
        if self.use_checkpoint and self.training:
            xi = checkpoint(self.ttm, xi, use_reentrant=False)
        else:
            xi = self.ttm(xi)
        return xi.reshape(-1)

    def forward(self, x):           # x : (B, C, T)
        feats = [self.forward_single(x[i]) for i in range(x.shape[0])]
        return torch.stack(feats)   # (B, feat_dim)


# ─── 4.  Projection Head ─────────────────────────────────────────────────────

class ProjectionHead(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int = 256, out_dim: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(self, x):
        return F.normalize(self.net(x), dim=-1)


# ─── 5.  NT-Xent Loss ────────────────────────────────────────────────────────

class NTXentLoss(nn.Module):
    def __init__(self, temperature: float = 0.5):
        super().__init__()
        self.eta = temperature

    def forward(self, z1: torch.Tensor, z2: torch.Tensor) -> torch.Tensor:
        N  = z1.shape[0]
        z  = torch.cat([z1, z2], dim=0)          # (2N, D)
        sim = z @ z.T / self.eta                  # (2N, 2N)
        mask = torch.eye(2 * N, dtype=torch.bool, device=z.device)
        sim.masked_fill_(mask, float('-inf'))
        labels = torch.cat([
            torch.arange(N, 2 * N, device=z.device),
            torch.arange(0, N,     device=z.device),
        ])
        return F.cross_entropy(sim, labels)


print("✅  Contrastive learning classes defined.")

✅  Contrastive learning classes defined.


In [10]:
# ─── 6.  Contrastive Pre-training ────────────────────────────────────────────
#
# Key fixes applied here:
#   FIX 6  : AMP (automatic mixed precision) — halves VRAM for activations
#   FIX 7  : Gradient accumulation (ACCUM_STEPS=4) — effective batch = 64
#             while only 16 samples live on GPU at once
#   FIX 8  : torch.cuda.empty_cache() after each accumulation step
#   FIX 9  : CL_BATCH_SIZE reduced from 64 → 16 (gradient accumulation
#             preserves the effective batch for NT-Xent negatives)
# ─────────────────────────────────────────────────────────────────────────────

CL_BATCH_SIZE  = 32      # FIX 9: was 64 — large batches are the primary OOM cause
ACCUM_STEPS    = 2       # FIX 7: effective batch = 16 * 4 = 64 (same as before)
CL_EPOCHS      = 5
CL_LR          = 3e-4
CL_TEMPERATURE = 0.5

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# --- build model (use_checkpoint=True enables gradient checkpointing) --------
full_model = L_SCLNet_EEGformer(
    input_channels  = 21,
    time_steps      = WIN_SAMPLES,
    num_classes     = 2,
    kernel_size     = 8,
    num_blocks      = 2,
    num_heads_rtm   = 6,
    num_heads_stm   = 6,
    num_heads_ttm   = 6,
    num_submatrices = 6,
    CF_second       = 2,
    use_checkpoint  = True,   # FIX 4: gradient checkpointing on STM + TTM
).to(device)

print(f'Total parameters: {sum(p.numel() for p in full_model.parameters() if p.requires_grad):,}')

# --- contrastive dataset ----------------------------------------------------
cl_dataset = ContrastiveEEGDataset(X_train, noise_alpha=0.0, noise_beta=0.1)
cl_loader  = DataLoader(
    cl_dataset,
    batch_size  = CL_BATCH_SIZE,
    shuffle     = True,
    num_workers = 2,
    pin_memory  = True,
    drop_last   = True,
)

# --- backbone + projection head ---------------------------------------------
backbone  = EEGformerBackbone(full_model).to(device)

with torch.no_grad():
    _dummy   = torch.zeros(1, 21, WIN_SAMPLES, device=device)
    _feat    = backbone(_dummy)
    feat_dim = _feat.shape[-1]
    print(f'Backbone feature dimension: {feat_dim:,}')

proj_head = ProjectionHead(in_dim=feat_dim, hidden_dim=512, out_dim=128).to(device)
nt_xent   = NTXentLoss(temperature=CL_TEMPERATURE)

cl_params = list(backbone.parameters()) + list(proj_head.parameters())
cl_optim  = torch.optim.AdamW(cl_params, lr=CL_LR, weight_decay=1e-4)
cl_sched  = torch.optim.lr_scheduler.CosineAnnealingLR(
    cl_optim, T_max=CL_EPOCHS * len(cl_loader), eta_min=1e-5
)

# FIX 6: GradScaler for AMP
scaler = torch.cuda.amp.GradScaler()

print(f'\n🔵  Starting contrastive pre-training ({CL_EPOCHS} epochs) …')
cl_history = []

for epoch in range(1, CL_EPOCHS + 1):
    backbone.train(); proj_head.train()
    running_loss = 0.0
    cl_optim.zero_grad()   # FIX 7: zero once before the accumulation loop

    for step, (v1, v2) in enumerate(tqdm(cl_loader, desc=f'CL Epoch {epoch:02d}', leave=False)):
        v1, v2 = v1.to(device), v2.to(device)

        # FIX 6: wrap forward pass in autocast (fp16 activations → ~half VRAM)
        with torch.cuda.amp.autocast():
            f1 = backbone(v1)             # (B, feat_dim)
            f2 = backbone(v2)             # ← was the OOM line; now fp16
            z1 = proj_head(f1)
            z2 = proj_head(f2)
            # FIX 7: divide loss by ACCUM_STEPS so gradients average correctly
            loss = nt_xent(z1, z2) / ACCUM_STEPS

        scaler.scale(loss).backward()
        running_loss += loss.item() * ACCUM_STEPS   # log unscaled value

        # FIX 7: optimizer step every ACCUM_STEPS mini-batches
        if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(cl_loader):
            scaler.unscale_(cl_optim)
            torch.nn.utils.clip_grad_norm_(cl_params, max_norm=1.0)
            scaler.step(cl_optim)
            scaler.update()
            cl_sched.step()
            cl_optim.zero_grad()
            # FIX 8: release any cached but unused CUDA memory
            torch.cuda.empty_cache()

    avg_cl_loss = running_loss / len(cl_loader)
    cl_history.append(avg_cl_loss)
    print(f'[CL Epoch {epoch:02d}]  NT-Xent loss = {avg_cl_loss:.4f}  '
          f'lr = {cl_sched.get_last_lr()[0]:.2e}')

print('✅  Contrastive pre-training complete.')
torch.save(backbone.state_dict(), 'pretrained_backbone.pth')

Using device: cuda
Total parameters: 3,471,081


/tmp/ipykernel_44/2686036120.py:68: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


Backbone feature dimension: 75,768

🔵  Starting contrastive pre-training (5 epochs) …


CL Epoch 01:   0%|          | 0/3906 [00:00<?, ?it/s]/tmp/ipykernel_44/2686036120.py:82: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
CL Epoch 01:   0%|          | 1/3906 [00:02<2:27:07,  2.26s/it]/tmp/ipykernel_44/2686036120.py:99: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  cl_sched.step()


[CL Epoch 01]  NT-Xent loss = 1.42  lr = 2.93e-04


[CL Epoch 02]  NT-Xent loss = 1.40  lr = 2.72e-04


[CL Epoch 03]  NT-Xent loss = 1.39  lr = 2.40e-04


[CL Epoch 04]  NT-Xent loss = 1.38  lr = 2.00e-04


[CL Epoch 05]  NT-Xent loss = 1.36  lr = 1.55e-04
✅  Contrastive pre-training complete.


In [11]:
# ─── 7.  Supervised Fine-tuning ───────────────────────────────────────────────
#
# The contrastive pre-training updated full_model.odcm/rtm/stm/ttm in-place.
# We now fine-tune the whole model (backbone + decoder) with AMP + grad accum.
#
# FIX 10: AMP + gradient accumulation applied here too.
# FIX 11: empty_cache() every accumulation step.
# ─────────────────────────────────────────────────────────────────────────────

FT_EPOCHS   = 5
FT_ACCUM    = 2     # effective batch = 16 * 2 = 32  (same as original BATCH_SIZE)

counts     = np.bincount(y_train)
class_wts  = torch.tensor(counts.sum() / (2 * counts), dtype=torch.float32).to(device)
criterion  = nn.CrossEntropyLoss(weight=class_wts)
ft_optim   = torch.optim.AdamW(full_model.parameters(), lr=1e-4, weight_decay=1e-4)
ft_sched   = torch.optim.lr_scheduler.OneCycleLR(
    ft_optim, max_lr=1e-4,
    # OneCycleLR counts optimizer steps, not mini-batch steps
    steps_per_epoch = math.ceil(len(train_loader) / FT_ACCUM),
    epochs          = FT_EPOCHS,
    pct_start       = 0.1,
)
ft_scaler  = torch.cuda.amp.GradScaler()   # FIX 10

best_val_acc    = 0.0
best_model_path = 'best_lsclnet_eegformer_cl.pth'
history         = {'train_loss': [], 'train_acc': [], 'val_acc': []}
LOG_EPOCHS      = {1, 4, 8}

print(f'\n🟢  Starting supervised fine-tuning ({FT_EPOCHS} epochs) …')

for epoch in range(1, FT_EPOCHS + 1):
    full_model.train()
    total_loss, all_preds, all_labels_list = 0.0, [], []
    ft_optim.zero_grad()

    for step, (xb, yb) in enumerate(tqdm(train_loader, desc=f'FT Epoch {epoch:02d}', leave=False)):
        xb, yb = xb.to(device), yb.to(device)

        with torch.cuda.amp.autocast():   # FIX 10
            logits = full_model(xb)
            loss   = criterion(logits, yb) / FT_ACCUM

        ft_scaler.scale(loss).backward()
        total_loss   += loss.item() * FT_ACCUM
        all_preds.extend(torch.argmax(logits.detach(), 1).cpu().numpy())
        all_labels_list.extend(yb.cpu().numpy())

        if (step + 1) % FT_ACCUM == 0 or (step + 1) == len(train_loader):
            ft_scaler.unscale_(ft_optim)
            torch.nn.utils.clip_grad_norm_(full_model.parameters(), max_norm=1.0)
            ft_scaler.step(ft_optim)
            ft_scaler.update()
            ft_sched.step()
            ft_optim.zero_grad()
            torch.cuda.empty_cache()   # FIX 11

    train_acc = accuracy_score(all_labels_list, all_preds)
    avg_loss  = total_loss / len(train_loader)
    history['train_loss'].append(avg_loss)
    history['train_acc'].append(train_acc)

    # ── validate (no AMP needed — inference is already fp32 on H100) ──────
    full_model.eval()
    val_preds, val_labels_list = [], []
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device)
            with torch.cuda.amp.autocast():
                logits_val = full_model(xb)
            val_preds.extend(torch.argmax(logits_val, 1).cpu().numpy())
            val_labels_list.extend(yb.numpy())

    val_acc = accuracy_score(val_labels_list, val_preds)
    val_f1  = f1_score(val_labels_list, val_preds, zero_division=0)
    history['val_acc'].append(val_acc)

    if epoch in LOG_EPOCHS:
        print(f'[FT Epoch {epoch:02d}]  loss={avg_loss:.4f}  '
              f'train_acc={train_acc*100:.2f}%  '
              f'val_acc={val_acc*100:.2f}%  '
              f'val_f1={val_f1:.4f}  '
              f'lr={ft_sched.get_last_lr()[0]:.2e}')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(full_model.state_dict(), best_model_path)
        if epoch in LOG_EPOCHS:
            print(f'  ✅  Best model saved  (val_acc={val_acc*100:.2f}%)')

print(f'\n🏆  Best val acc after fine-tuning: {best_val_acc*100:.2f}%')

/tmp/ipykernel_44/1304085335.py:24: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  ft_scaler  = torch.cuda.amp.GradScaler()   # FIX 10



🟢  Starting supervised fine-tuning (5 epochs) …


FT Epoch 01:   0%|          | 0/7812 [00:00<?, ?it/s]/tmp/ipykernel_44/1304085335.py:41: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():   # FIX 10
FT Epoch 01:   0%|          | 1/7812 [00:02<4:32:59,  2.10s/it]/tmp/ipykernel_44/1304085335.py:55: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  ft_sched.step()
/tmp/ipykernel_44/1304085335.py:70: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


[FT Epoch 01]  loss=0.6821  train_acc=72.2%  val_acc=72.01%  val_f1=0.7848  lr=9.70e-05
  ✅  Best model saved  (val_acc=72.01%)


FT Epoch 02:   0%|          | 0/7812 [00:00<?, ?it/s]/tmp/ipykernel_44/1304085335.py:41: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():   # FIX 10
/tmp/ipykernel_44/1304085335.py:70: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
FT Epoch 03:   0%|          | 0/7812 [00:00<?, ?it/s]/tmp/ipykernel_44/1304085335.py:41: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():   # FIX 10
/tmp/ipykernel_44/1304085335.py:70: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
FT Epoch 04:   0%|          | 0/7812 [00:00<?, ?it/s]/tmp/ipykernel_44/1304085335.py:41: FutureWarni

[FT Epoch 04]  loss=0.4251  train_acc=84.2%  val_acc=83.01%  val_f1=0.7948  lr=1.17e-05


FT Epoch 05:   0%|          | 0/7812 [00:00<?, ?it/s]/tmp/ipykernel_44/1304085335.py:41: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():   # FIX 10
/tmp/ipykernel_44/1304085335.py:70: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():



🏆  Best val acc after fine-tuning: 83.01%


In [12]:
# ─── Test set evaluation ──────────────────────────────────────────────────────
model = full_model
model.load_state_dict(torch.load('best_lsclnet_eegformer_cl.pth', map_location=device))
model.eval()

test_preds, test_labels_list = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        with torch.cuda.amp.autocast():
            test_preds.extend(torch.argmax(model(xb), 1).cpu().numpy())
        test_labels_list.extend(yb.numpy())

print('📊  Test Set Results')
print(f'   Accuracy  : {accuracy_score(test_labels_list, test_preds)*100:.2f}%')
print(f'   F1 Score  : {f1_score(test_labels_list, test_preds):.4f}')
print(f'   Precision : {precision_score(test_labels_list, test_preds):.4f}')
print(f'   Recall    : {recall_score(test_labels_list, test_preds):.4f}')
print()
print('Confusion Matrix:')
print(confusion_matrix(test_labels_list, test_preds))

/tmp/ipykernel_44/4257555512.py:10: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


📊  Test Set Results
   Accuracy  : 85.0300%
   F1 Score  : 0.8405
   Precision : 0.8400
   Recall    : 0.8610

Confusion Matrix:
[[ 7911  2206]
  [ 1755 10660]]
